In [45]:
pip install opencv-python scikit-image scikit-learn joblib numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
!pip install ultralytics opencv-python


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import cv2
import numpy as np
import os
import glob
from ultralytics import YOLO

# --- CẤU HÌNH ---
INPUT_FOLDER = 'input_images'
OUTPUT_FOLDER = 'output_final'
CONFIDENCE_THRESHOLD = 0.4
CORNER_THRESHOLD = 8

# Tham số Gaussian Blur (Quan trọng)
# (3, 3) là làm mờ nhẹ, (5, 5) là trung bình, (7, 7) là mờ mạnh
GAUSSIAN_KERNEL_SIZE = (5, 5)

prediction_cache = [] 

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# --- CÁC HÀM XỬ LÝ ---

def preprocess_pipeline(image):
    """
    Quy trình Tiền xử lý kết hợp:
    1. CLAHE: Cân bằng sáng
    2. Gaussian Blur: Khử nhiễu sinh ra do CLAHE hoặc do cảm biến camera
    """
    # Bước 1: Áp dụng CLAHE
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # Bước 2: Áp dụng Gaussian Blur lên ảnh đã CLAHE
    # sigmaX=0 nghĩa là để OpenCV tự tính toán dựa trên kernel size
    blurred_img = cv2.GaussianBlur(enhanced_img, GAUSSIAN_KERNEL_SIZE, 0)

    return blurred_img

def verify_with_corners(image_roi_gray):
    # (Giữ nguyên hàm đếm góc cũ)
    corners = cv2.goodFeaturesToTrack(image_roi_gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
    return len(corners) if corners is not None else 0

# --- CHƯƠNG TRÌNH CHÍNH ---

print("Đang tải model YOLOv8...")
model = YOLO('yolov8n.pt')
vehicle_classes = [2, 3, 5, 7]

image_files = glob.glob(os.path.join(INPUT_FOLDER, '*.*'))

print(f"Bắt đầu xử lý với Pipeline: CLAHE -> Gaussian Blur -> YOLO -> Corner Filter")

for img_path in image_files:
    filename = os.path.basename(img_path)
    original_img = cv2.imread(img_path)
    if original_img is None: continue

    # --- BƯỚC 1 & 2: TIỀN XỬ LÝ KÉP ---
    # Ảnh này sẽ dùng để đưa vào YOLO (đã sáng và mịn)
    input_for_yolo = preprocess_pipeline(original_img)

    # --- BƯỚC 3: YOLO DETECTION ---
    results = model(input_for_yolo, verbose=False)

    detected_count = 0
    final_output = original_img.copy() # Vẽ lên ảnh gốc sắc nét
    
    current_boxes = []

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            if cls in vehicle_classes and conf > CONFIDENCE_THRESHOLD:
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                h, w, _ = original_img.shape
                # Cắt ROI từ ảnh GỐC (để đếm góc chính xác nhất, không dùng ảnh mờ)
                roi = original_img[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
                if roi.size == 0: continue

                # --- BƯỚC 4: LỌC NHIỄU BẰNG GÓC ---
                roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
                num_corners = verify_with_corners(roi_gray)

                if num_corners < CORNER_THRESHOLD:
                    continue

                detected_count += 1
                

                # Vẽ kết quả
                class_name = model.names[cls] if cls in model.names else str(cls)
                cv2.rectangle(final_output, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"{class_name} ({num_corners})"
                cv2.putText(final_output, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                current_boxes.append([x1, y1, x2, y2])

    prediction_cache.append({
        'filename': filename,
        'shape': (h, w),
        'boxes': current_boxes
    })
    
    # Lưu và in kết quả (Giữ nguyên)
    if detected_count > 0:
        save_path = os.path.join(OUTPUT_FOLDER, "detected_" + filename)
        cv2.imwrite(save_path, final_output)
        print(f"--> {filename}: Đã phát hiện {detected_count} xe.")
    else:
        print(f"--> {filename}: Không tìm thấy xe.")

print("\nHoàn tất!")

Đang tải model YOLOv8...
Bắt đầu xử lý với Pipeline: CLAHE -> Gaussian Blur -> YOLO -> Corner Filter
--> 00110.jpg: Đã phát hiện 1 xe.
--> 00262.jpg: Đã phát hiện 1 xe.
--> 00790.jpg: Đã phát hiện 1 xe.
--> 00826.jpg: Đã phát hiện 1 xe.
--> 01000_GMC Savana Van 2012.jpg: Đã phát hiện 1 xe.
--> 01444.jpg: Đã phát hiện 1 xe.
--> 01455.jpg: Đã phát hiện 1 xe.
--> 01890.jpg: Đã phát hiện 1 xe.
--> 02136.jpg: Đã phát hiện 1 xe.
--> 02321.jpg: Đã phát hiện 2 xe.
--> 02358_Nissan NV Passenger Van 2012.jpg: Đã phát hiện 1 xe.
--> 02381.jpg: Đã phát hiện 1 xe.
--> 02504.jpg: Đã phát hiện 1 xe.
--> 02564_Chevrolet Express Van 2007.jpg: Đã phát hiện 1 xe.
--> 02743.jpg: Đã phát hiện 1 xe.
--> 02878_GMC Savana Van 2012.jpg: Đã phát hiện 1 xe.
--> 03137_Nissan NV Passenger Van 2012.jpg: Đã phát hiện 2 xe.
--> 03206.jpg: Đã phát hiện 1 xe.
--> 03218_Mercedes-Benz Sprinter Van 2012.jpg: Đã phát hiện 1 xe.
--> 03363_Dodge Sprinter Cargo Van 2009.jpg: Đã phát hiện 1 xe.
--> 04129.jpg: Đã phát hiện 1 xe

In [6]:
# --- CẤU HÌNH ĐÁNH GIÁ ---
LABELS_FOLDER = 'labels' 
IOU_THRESHOLD = 0.5

# --- HÀM HỖ TRỢ TÍNH TOÁN ---
def calculate_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    boxBArea = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)

def load_ground_truth(label_path, img_w, img_h):
    boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                p = list(map(float, line.strip().split()))
                # p[0] là class_id. Nếu cần lọc class ID trong nhãn thì thêm if p[0] in ...
                x_c, y_c, w, h_box = p[1], p[2], p[3], p[4]
                x1, y1 = int((x_c - w/2)*img_w), int((y_c - h_box/2)*img_h)
                x2, y2 = int((x_c + w/2)*img_w), int((y_c + h_box/2)*img_h)
                boxes.append([x1, y1, x2, y2])
    return boxes

# --- CHẠY ĐÁNH GIÁ TỪ CACHE ---
if not prediction_cache:
    print("Lỗi: Hãy chạy Cell 1 trước để có dữ liệu!")
else:
    tp, fp, fn = 0, 0, 0
    iou_list = []

    for item in prediction_cache:
        # Lấy dữ liệu từ Cell 1
        filename = item['filename']
        pred_boxes = item['boxes']
        h, w = item['shape']
        
        # Load đáp án tương ứng
        label_path = os.path.join(LABELS_FOLDER, os.path.splitext(filename)[0] + ".txt")
        gt_boxes = load_ground_truth(label_path, w, h)
        
        matched_gt = [False] * len(gt_boxes)
        
        # So khớp
        for pred in pred_boxes:
            best_iou = 0
            best_idx = -1
            for i, gt in enumerate(gt_boxes):
                iou = calculate_iou(pred, gt)
                if iou > best_iou:
                    best_iou, best_idx = iou, i
            
            if best_iou >= IOU_THRESHOLD:
                if not matched_gt[best_idx]:
                    tp += 1
                    matched_gt[best_idx] = True
                    iou_list.append(best_iou)
                else:
                    fp += 1 # Duplicate
            else:
                fp += 1 # Wrong location
        
        fn += matched_gt.count(False)

    # Tính Metrics
    eps = 1e-6
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    m_iou = sum(iou_list)/len(iou_list) if iou_list else 0
    
    print("\n" + "="*40)
    print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ({len(prediction_cache)} ẢNH)")
    print("="*40)
    print(f"True Positives (Bắt đúng):  {tp}")
    print(f"False Positives (Bắt nhầm): {fp}")
    print(f"False Negatives (Bỏ sót):   {fn}")
    print("-" * 40)
    print(f"PRECISION:   {precision:.2%}")
    print(f"RECALL:      {recall:.2%}")
    print(f"F1-SCORE:    {f1:.2%}")
    print(f"AVERAGE IOU: {m_iou:.2%}")
    print("="*40)
    


KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (930 ẢNH)
True Positives (Bắt đúng):  1154
False Positives (Bắt nhầm): 125
False Negatives (Bỏ sót):   191
----------------------------------------
PRECISION:   90.23%
RECALL:      85.80%
F1-SCORE:    87.96%
AVERAGE IOU: 92.22%


In [8]:
import cv2
import numpy as np
import os
import glob
from collections import Counter
from ultralytics import YOLO

# --- CẤU HÌNH ---
IMAGES_FOLDER = 'input_images'
LABELS_FOLDER = 'labels'
MODEL_PATH = 'yolov8n.pt'

# --- THAM SỐ TINH CHỈNH ---
CONFIDENCE_THRESHOLD = 0.4
CORNER_THRESHOLD = 5
IOU_THRESHOLD = 0.5  # Ngưỡng để xác định True Positive cho việc tính mAP@50
GAUSSIAN_KERNEL = (5, 5)

# ID các lớp xe (COCO format: 2=Car, 3=Motorcycle, 5=Bus, 7=Truck)
VEHICLE_CLASSES = [2, 3, 5, 7]
# Tên các lớp để in báo cáo
CLASS_NAMES = {2: 'Car', 3: 'Motorcycle', 5: 'Bus', 7: 'Truck'}

# --- CÁC HÀM XỬ LÝ (PIPELINE CỦA BẠN) ---

def preprocess_pipeline(image):
    """Tiền xử lý: CLAHE + Gaussian Blur"""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    blurred = cv2.GaussianBlur(enhanced, GAUSSIAN_KERNEL, 0)
    return blurred

def count_corners(image_roi):
    """Hậu kiểm: Đếm góc"""
    if image_roi.size == 0: return 0
    gray = cv2.cvtColor(image_roi, cv2.COLOR_BGR2GRAY)
    # Dùng Harris hoặc Shi-Tomasi đều được (ở đây dùng Shi-Tomasi cho ổn định)
    corners = cv2.goodFeaturesToTrack(gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
    return len(corners) if corners is not None else 0

# --- CÁC HÀM HỖ TRỢ TÍNH mAP ---

def calculate_iou_vectorized(boxA, boxB):
    """Tính IoU giữa 1 box và 1 box khác"""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea + 1e-6)

def load_ground_truth(label_path, img_w, img_h, image_id):
    """Đọc file label và trả về danh sách GT kèm ID ảnh"""
    boxes = []
    if not os.path.exists(label_path): return []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = list(map(float, line.strip().split()))
            cls_id = int(parts[0])
            if cls_id in VEHICLE_CLASSES:
                x_c, y_c, w, h = parts[1], parts[2], parts[3], parts[4]
                x1 = int((x_c - w/2) * img_w)
                y1 = int((y_c - h/2) * img_h)
                x2 = int((x_c + w/2) * img_w)
                y2 = int((y_c + h/2) * img_h)
                # Format: [image_id, class_id, x1, y1, x2, y2]
                boxes.append([image_id, cls_id, x1, y1, x2, y2])
    return boxes

def calculate_ap_per_class(detections, ground_truths, iou_threshold=0.5):
    """
    Tính Average Precision (AP) cho một class cụ thể.
    """
    # Nếu không có GT nào cho class này -> AP = 0
    if len(ground_truths) == 0:
        return 0, 0, 0 # AP, Precision, Recall

    # Sắp xếp dự đoán theo độ tin cậy giảm dần (Quan trọng cho mAP)
    detections.sort(key=lambda x: x[2], reverse=True)

    TP = np.zeros(len(detections))
    FP = np.zeros(len(detections))

    # Theo dõi xem GT nào đã được detect rồi để tránh tính trùng
    gt_matched = {i: np.zeros(len([g for g in ground_truths if g[0] == i])) for i in set(g[0] for g in ground_truths)}

    # Lặp qua từng dự đoán
    for i, detection in enumerate(detections):
        img_id = detection[0]
        box_pred = detection[3:]

        # Lấy tất cả GT thuộc ảnh này
        gts_in_image = [g for g in ground_truths if g[0] == img_id]

        best_iou = 0
        best_gt_idx = -1

        # Tìm GT khớp nhất
        for j, gt in enumerate(gts_in_image):
            iou = calculate_iou_vectorized(box_pred, gt[2:])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        # Kiểm tra ngưỡng IoU
        if best_iou >= iou_threshold:
            # Kiểm tra xem GT này đã bị bắt chưa
            if gt_matched[img_id][best_gt_idx] == 0:
                TP[i] = 1 # Đúng
                gt_matched[img_id][best_gt_idx] = 1
            else:
                FP[i] = 1 # Sai (Bắt trùng lặp)
        else:
            FP[i] = 1 # Sai (Bắt trượt hoặc vào nền)

    # Tính Cumulative Sum (Tích lũy)
    TP_cumsum = np.cumsum(TP)
    FP_cumsum = np.cumsum(FP)

    epsilon = 1e-6
    recalls = TP_cumsum / (len(ground_truths) + epsilon)
    precisions = TP_cumsum / (TP_cumsum + FP_cumsum + epsilon)

    # Tính AP bằng cách tích phân đường cong P-R (Area Under Curve)
    # Thêm điểm 0 và 1 vào đầu/cuối để tính diện tích chuẩn hơn
    recalls = np.concatenate(([0.0], recalls, [1.0]))
    precisions = np.concatenate(([0.0], precisions, [0.0]))

    # Làm mượt đường cong (Interpolation)
    for i in range(len(precisions) - 1, 0, -1):
        precisions[i-1] = max(precisions[i-1], precisions[i])

    # Tính diện tích
    indices = np.where(recalls[1:] != recalls[:-1])[0]
    ap = np.sum((recalls[indices + 1] - recalls[indices]) * precisions[indices + 1])

    # Trả về AP và Precision/Recall cuối cùng
    final_prec = precisions[-2] if len(precisions) > 1 else 0
    final_rec = recalls[-2] if len(recalls) > 1 else 0

    return ap, final_prec, final_rec

# --- MAIN EVALUATION ---

def run_evaluation():
    print(f"Đang tải model {MODEL_PATH}...")
    model = YOLO(MODEL_PATH)

    image_paths = glob.glob(os.path.join(IMAGES_FOLDER, '*.*'))
    valid_exts = ['.jpg', '.jpeg', '.png']
    image_paths = [p for p in image_paths if os.path.splitext(p)[1].lower() in valid_exts]

    print(f"Tìm thấy {len(image_paths)} ảnh. Đang chạy Hybrid Pipeline và thu thập dữ liệu...")

    # Danh sách chứa toàn bộ dữ liệu để tính mAP
    all_detections = []    # [image_id, class_id, conf, x1, y1, x2, y2]
    all_ground_truths = [] # [image_id, class_id, x1, y1, x2, y2]

    for idx, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        label_name = os.path.splitext(filename)[0] + ".txt"
        label_path = os.path.join(LABELS_FOLDER, label_name)
        image_id = idx # Dùng index làm ID cho ảnh

        # 1. Load Ground Truth
        original_img = cv2.imread(img_path)
        if original_img is None: continue
        h, w, _ = original_img.shape

        gts = load_ground_truth(label_path, w, h, image_id)
        all_ground_truths.extend(gts)

        # 2. Hybrid Pipeline Prediction
        # B1: Pre-process
        processed_img = preprocess_pipeline(original_img)
        # B2: YOLO Predict
        results = model(processed_img, verbose=False)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])

                if cls in VEHICLE_CLASSES and conf > CONFIDENCE_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    # B3: Corner Filter
                    roi = original_img[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
                    corners = count_corners(roi)

                    if corners >= CORNER_THRESHOLD:
                        # Lưu kết quả để tính mAP sau này
                        all_detections.append([image_id, cls, conf, x1, y1, x2, y2])


    print("\n" + "="*50)
    print(f"KẾT QUẢ ĐÁNH GIÁ (mAP@50)")
    print("="*50)
    print(f"{'Class':<15} | {'AP@50':<10} | {'Precision':<10} | {'Recall':<10}")
    print("-" * 50)

    aps = []

    # Tính AP cho từng Class
    for cls_id in VEHICLE_CLASSES:
        # Lọc ra detections và GTs của class hiện tại
        cls_dets = [d for d in all_detections if d[1] == cls_id]
        cls_gts = [g for g in all_ground_truths if g[1] == cls_id]

        ap, prec, rec = calculate_ap_per_class(cls_dets, cls_gts, iou_threshold=0.5)
        aps.append(ap)

        cls_name = CLASS_NAMES.get(cls_id, str(cls_id))
        print(f"{cls_name:<15} | {ap:.4f}     | {prec:.4f}     | {rec:.4f}")

    mAP = sum(aps) / len(aps) if aps else 0

    print("-" * 50)
    print(f"mAP@50 (All):   {mAP:.4f}")
    print("="*50)

if __name__ == "__main__":
    run_evaluation()

Đang tải model yolov8n.pt...
Tìm thấy 930 ảnh. Đang chạy Hybrid Pipeline và thu thập dữ liệu...

KẾT QUẢ ĐÁNH GIÁ (mAP@50)
Class           | AP@50      | Precision  | Recall    
--------------------------------------------------
Car             | 0.7472     | 0.6184     | 0.8304
Motorcycle      | 0.4731     | 0.9538     | 0.4749
Bus             | 0.9014     | 0.6583     | 0.9375
Truck           | 0.6268     | 0.6947     | 0.6889
--------------------------------------------------
mAP@50 (All):   0.6871


In [ ]:
import cv2
import numpy as np
import os
import glob
from ultralytics import YOLO

# --- CẤU HÌNH ---
INPUT_FOLDER = 'input_images'
OUTPUT_BASE_FOLDER = 'report_outputs'
CONFIDENCE_THRESHOLD = 0.4
CORNER_THRESHOLD = 8
GAUSSIAN_KERNEL_SIZE = (5, 5)

# [MỚI] Thêm folder '2b_yolo_raw' để chứa ảnh YOLO gốc
DEBUG_FOLDERS = {
    'clahe': os.path.join(OUTPUT_BASE_FOLDER, '1_clahe'),
    'blur': os.path.join(OUTPUT_BASE_FOLDER, '2_blur'),
    'yolo_raw': os.path.join(OUTPUT_BASE_FOLDER, '2b_yolo_raw'), 
    'roi': os.path.join(OUTPUT_BASE_FOLDER, '3_roi_corners'),
    'final': os.path.join(OUTPUT_BASE_FOLDER, '4_final')
}

for path in DEBUG_FOLDERS.values():
    if not os.path.exists(path):
        os.makedirs(path)

# --- CÁC HÀM XỬ LÝ (Giữ nguyên) ---

def preprocess_pipeline(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    clahe_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    blurred_img = cv2.GaussianBlur(clahe_img, GAUSSIAN_KERNEL_SIZE, 0)
    return clahe_img, blurred_img

def analyze_and_draw_corners(roi_color):
    roi_gray = cv2.cvtColor(roi_color, cv2.COLOR_BGR2GRAY)
    corners = cv2.goodFeaturesToTrack(roi_gray, maxCorners=100, qualityLevel=0.01, minDistance=10)
    count = 0
    roi_visual = roi_color.copy()
    if corners is not None:
        count = len(corners)
        corners = np.int32(corners)
        for i in corners:
            x, y = i.ravel()
            cv2.circle(roi_visual, (x, y), 3, (0, 0, 255), -1) 
    return count, roi_visual

# --- HÀM VẼ KHUNG THÔNG MINH (Dùng chung cho cả YOLO Raw và Final) ---
def draw_smart_box(img, x1, y1, x2, y2, label, color=(0, 255, 0)):
    h_img, w_img = img.shape[:2]
    font_scale = max(0.5, w_img / 1000)
    thickness = max(1, int(w_img / 500))
    
    # Tính kích thước chữ
    (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

    # Xử lý vị trí chữ (tránh bị tràn lên trên)
    if y1 - text_h - 10 < 0:
        text_y = y1 + text_h + 10
    else:
        text_y = y1 - 10

    # Vẽ nền chữ
    cv2.rectangle(img, (x1, text_y - text_h - 5), (x1 + text_w, text_y + 5), color, -1)
    # Vẽ khung
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    # Vẽ chữ (màu đen hoặc trắng tùy độ tương phản, ở đây chọn Đen trên nền màu)
    cv2.putText(img, label, (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)

# --- CHƯƠNG TRÌNH CHÍNH ---

model_path = 'yolov8n.pt' 
print(f"Đang tải model {model_path}...")
model = YOLO(model_path)
vehicle_classes = [2, 3, 5, 7] 

image_files = glob.glob(os.path.join(INPUT_FOLDER, '*.*'))

print(f"Bắt đầu xử lý...")

for img_path in image_files:
    filename = os.path.basename(img_path)
    name_no_ext = os.path.splitext(filename)[0]
    
    original_img = cv2.imread(img_path)
    if original_img is None: continue

    # B1 & B2: Preprocessing
    clahe_img, input_for_yolo = preprocess_pipeline(original_img)
    cv2.imwrite(os.path.join(DEBUG_FOLDERS['clahe'], f"clahe_{filename}"), clahe_img)
    cv2.imwrite(os.path.join(DEBUG_FOLDERS['blur'], f"blur_{filename}"), input_for_yolo)

    # B3: YOLO Detection
    results = model(input_for_yolo, verbose=False)

    detected_count = 0
    final_output = original_img.copy()
    
    # [MỚI] Tạo ảnh YOLO Raw (chưa lọc góc)
    # Mình dùng màu Xanh Dương (255, 0, 0) cho YOLO thuần để phân biệt với kết quả cuối
    yolo_raw_output = original_img.copy() 
    yolo_raw_count = 0 

    for r in results:
        for i, box in enumerate(r.boxes):
            cls = int(box.cls[0])
            conf = float(box.conf[0])

            if cls in vehicle_classes and conf > CONFIDENCE_THRESHOLD:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                class_name = model.names[cls] if hasattr(model, 'names') else str(cls)
                
                # --- [MỚI] LƯU ẢNH YOLO RAW (Bước này làm TRƯỚC khi lọc) ---
                # Vẽ tất cả những gì YOLO thấy lên ảnh 'yolo_raw_output'
                raw_label = f"{class_name} {conf:.2f}"
                draw_smart_box(yolo_raw_output, x1, y1, x2, y2, raw_label, color=(255, 100, 0)) # Màu Xanh Dương/Cam
                yolo_raw_count += 1
                
                # --- B4: CẮT ẢNH & ĐẾM GÓC ---
                h, w, _ = original_img.shape
                roi = original_img[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
                if roi.size == 0: continue

                num_corners, roi_with_corners = analyze_and_draw_corners(roi)
                roi_filename = f"{name_no_ext}_box{i}_{class_name}_corners{num_corners}.jpg"
                cv2.imwrite(os.path.join(DEBUG_FOLDERS['roi'], roi_filename), roi_with_corners)

                # --- B5: LỌC NHIỄU (Corner Filter) ---
                if num_corners < CORNER_THRESHOLD:
                    continue 

                detected_count += 1
                
                # Vẽ kết quả cuối cùng (Final) lên ảnh 'final_output'
                # Dùng màu Xanh Lá (0, 255, 0) cho xe đã xác nhận
                final_label = f"{class_name} ({num_corners})"
                draw_smart_box(final_output, x1, y1, x2, y2, final_label, color=(0, 255, 0))

    # Lưu ảnh YOLO Raw
    if yolo_raw_count > 0:
        cv2.imwrite(os.path.join(DEBUG_FOLDERS['yolo_raw'], f"yolo_{filename}"), yolo_raw_output)

    # Lưu ảnh Final
    if detected_count > 0:
        cv2.imwrite(os.path.join(DEBUG_FOLDERS['final'], f"final_{filename}"), final_output)
        print(f"--> {filename}: YOLO thấy {yolo_raw_count} box -> Sau khi lọc còn {detected_count} xe.")
    else:
        print(f"--> {filename}: Không tìm thấy xe.")


Đang tải model yolov8n.pt...
Bắt đầu xử lý...
--> 00110.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 00262.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 00790.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 00826.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 01000_GMC Savana Van 2012.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 01444.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 01455.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 01890.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02136.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02321.jpg: YOLO thấy 2 box -> Sau khi lọc còn 2 xe.
--> 02358_Nissan NV Passenger Van 2012.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02381.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02504.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02564_Chevrolet Express Van 2007.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02743.jpg: YOLO thấy 1 box -> Sau khi lọc còn 1 xe.
--> 02878_GMC Savana Van 2012.jpg: YOL